# Data Preparation: Part 5 - Flight Phase

## 1. Import Packages & Define Custom Functions

In [1]:
import pandas as pd
import re

In [2]:
# Using custom function from Michael Albert's class

def summarize_dataframe(df):
    missing_values = pd.concat([pd.DataFrame(df.columns, columns=['Variable Name']), 
                      pd.DataFrame(df.dtypes.values.reshape([-1,1]), columns=['Data Type']),
                      pd.DataFrame(df.isnull().sum().values, columns=['Missing Values']), 
                      pd.DataFrame([df[name].nunique() for name in df.columns], columns=['Unique Values'])], 
                     axis=1).set_index('Variable Name')
    return pd.concat([missing_values, df.describe(include='all').transpose()], axis=1).fillna("")

## 2. Import Raw Data

In [3]:
full = pd.read_csv('Raw Input Files/asrs_full.csv')

In [4]:
# Create dataframe with just flight_phase
fp = full.flight_phase.astype(str)

## 3. Clean Data

### 3.1. Identify Unique Categories

In [5]:
disagg_fp = []
for phrase in fp:
    disagg_fp.extend([s.strip() for s in re.split(r';', phrase)])

In [6]:
disagg_fp = pd.Series(disagg_fp)

In [7]:
# Counts
value_counts = disagg_fp.value_counts().reset_index()
value_counts.columns = ["Phase", "Count"]

### 3.2. Group Uncommon Categories Into Other

In [8]:
# create a copy of the data
dummy_full = full.copy()

dummy_full['flight_phase']=dummy_full['flight_phase'].fillna('nan')

# Making sure to cut out anything from the others list that is not marked 'other'
others = list(value_counts['Phase'][12:])
others = [x for x in others if x not in ['Ground / Preflight (UAS)', 'Return to Home (UAS)']]
others_sorted = sorted(others, key=len, reverse=True)

for value in others_sorted:
    dummy_full['flight_phase'] = dummy_full['flight_phase'].str.replace(value, 'other', regex=False)

### 3.3. Create Dummy Columns

In [9]:
# Obtain dummy columns
dummy_full['flight_phase'] = dummy_full['flight_phase'].str.replace(' ', '', regex=False)
dummies = dummy_full['flight_phase'].str.get_dummies(sep=';')

In [10]:
# Rename variables to include prefix "phase_"
rename = dummies.columns
rename_to = []
for x in rename:
    rename_to.append("phase_" + x)

dummies.columns=rename_to

In [11]:
# Concat with original data
dummy_full = pd.concat([full.acn, full.flight_phase, dummies], axis=1)

In [12]:
with pd.option_context('display.max_rows', None):
    display(summarize_dataframe(dummy_full))

/tmp/ipykernel_2103/3979652495.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.concat([missing_values, df.describe(include='all').transpose()], axis=1).fillna("")


,Data Type,Missing Values,Unique Values,count,unique,top,freq,mean,std,min,25%,50%,75%,max
acn,int64,0,33723,33723.0,,,,1828811.478783,192110.173434,1507557.0,1673862.5,1806271.0,1981117.5,2296796.0
flight_phase,object,903,347,32820.0,347,Cruise,5064,,,,,,,
phase_Climb,int64,0,2,33723.0,,,,0.10782,0.310157,0.0,0.0,0.0,0.0,1.0
phase_Cruise,int64,0,2,33723.0,,,,0.160187,0.366785,0.0,0.0,0.0,0.0,1.0
phase_Descent,int64,0,2,33723.0,,,,0.086499,0.281103,0.0,0.0,0.0,0.0,1.0
phase_FinalApproach,int64,0,2,33723.0,,,,0.093408,0.291008,0.0,0.0,0.0,0.0,1.0
phase_Ground/Preflight(UAS),int64,0,2,33723.0,,,,0.000326,0.018058,0.0,0.0,0.0,0.0,1.0
phase_Hovering(UAS),int64,0,2,33723.0,,,,0.002817,0.053002,0.0,0.0,0.0,0.0,1.0
phase_InitialApproach,int64,0,2,33723.0,,,,0.104765,0.306255,0.0,0.0,0.0,0.0,1.0
phase_InitialClimb,int64,0,2,33723.0,,,,0.063992,0.244742,0.0,0.0,0.0,0.0,1.0


## 4. Export Data

In [13]:
# Export
#dummy_full.to_csv('Output Files/Data 5 - flight_phase clean.csv', index=False)